# LLMs & Transformers
by AI@UCI

** ENSURE YOU ARE RUNNING THIS IN AN ENVIRONMENT WITH THE REQUIRED PACKAGES **

## Overview

In this notebook we'll explore the fundamentals of Large Language Models (LLMs) and the Transformer architecture that powers them. We'll cover:

- **Tokenization** — how raw text is converted into numbers a model can process
- **Embeddings** — how token IDs become rich vector representations
- **Attention Mechanism** — the core innovation of Transformers
- **Text Generation** — running a real LLM (GPT-2) to generate text


In [ ]:
! pip install transformers torch numpy matplotlib

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from transformers import AutoTokenizer, AutoModel, pipeline

## What is a Transformer?

The **Transformer** architecture was introduced in the 2017 paper *"Attention Is All You Need"* (Vaswani et al.) and has since become the backbone of nearly every modern LLM.

Key ideas:
- Instead of processing words one at a time (like RNNs), Transformers look at **all words simultaneously**.
- The magic ingredient is **self-attention**: each word can "attend" to every other word in the sequence.
- **Large Language Models** (GPT, BERT, LLaMA, Claude, etc.) are Transformers trained on massive text datasets.

```
Text  -->  Tokenizer  -->  Embeddings  -->  Transformer Layers  -->  Output
 "Hello"     [15496]       [[0.3, -0.1, ...]]    (attention x N)    next token
```

> **Discussion:** What do you think makes this architecture better than processing words one by one? What kinds of language patterns might be hard for a sequential model to capture?


## 1. Tokenization

Before a model can process text, the text must be converted into numbers. A **tokenizer** splits text into *tokens* (roughly, subwords) and maps each to an integer ID.

> **Discussion:** Why do you think models use *subword* tokens instead of whole words or individual characters? What are the tradeoffs?


In [ ]:
# Load the GPT-2 tokenizer
tokenizer = AutoTokenizer.from_pretrained("gpt2")

text = "Large language models are transforming AI research."

# Encode: text -> list of token IDs
token_ids = tokenizer.encode(text)
print("Token IDs:", token_ids)

# Convert IDs back to readable token strings
tokens = tokenizer.convert_ids_to_tokens(token_ids)
print("Tokens:   ", tokens)

print("\nVocabulary size:", tokenizer.vocab_size)

> **Question:** Look at the token list above. Can you spot any words that got split into multiple tokens? Why might "transforming" be split differently from "AI"?


In [ ]:
# Decode: token IDs -> text (round-trip check)
decoded = tokenizer.decode(token_ids)
print("Decoded text:", decoded)

# Visualize token lengths for each word
words = text.split()
word_token_counts = []
for word in words:
    count = len(tokenizer.encode(" " + word, add_special_tokens=False))
    word_token_counts.append(count)

plt.bar(words, word_token_counts, color='steelblue')
plt.title("Number of Tokens per Word")
plt.ylabel("Token count")
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.show()

> **Challenge:** Try tokenizing a sentence with unusual or rare words (e.g., a technical term or a made-up word). How many tokens does each word get? What does this tell you about how the tokenizer handles unknown vocabulary?


In [ ]:
# Try it yourself: tokenize a custom sentence
custom_text = ""  # <-- fill in your sentence here
custom_ids = tokenizer.encode(custom_text)
print("Token IDs:", custom_ids)
print("Tokens:   ", tokenizer.convert_ids_to_tokens(custom_ids))

## 2. Embeddings

Token IDs are just integers — they carry no semantic meaning by themselves. **Embeddings** map each token ID to a dense vector of real numbers.

- The embedding matrix has shape `(vocab_size, embedding_dim)`.
- Each row is a learned vector that captures meaning. Similar words end up with similar vectors.
- GPT-2 uses an embedding dimension of **768**.

> **Discussion:** If the embedding dimension is 768, what does that mean geometrically? How might two words with similar meanings end up with similar vectors?


In [ ]:
import torch

# Load the GPT-2 model to inspect its embedding layer
model = AutoModel.from_pretrained("gpt2")
embedding_layer = model.wte  # word token embeddings

print("Embedding matrix shape:", embedding_layer.weight.shape)
# Shape: (vocab_size, embedding_dim) = (50257, 768)

sample_embedding = embedding_layer.weight[15496].detach().numpy()  # token for "hello"
print("\nEmbedding vector for 'hello' (first 10 dims):", sample_embedding[:10])
print("Embedding dimension:", len(sample_embedding))

In [ ]:
# Cosine similarity between two embeddings
def cosine_similarity(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

# Compare embeddings of semantically related vs unrelated word pairs
word_pairs = [("king", "queen"), ("dog", "cat"), ("computer", "table")]

for w1, w2 in word_pairs:
    id1 = tokenizer.encode(" " + w1, add_special_tokens=False)[0]
    id2 = tokenizer.encode(" " + w2, add_special_tokens=False)[0]
    e1 = embedding_layer.weight[id1].detach().numpy()
    e2 = embedding_layer.weight[id2].detach().numpy()
    sim = cosine_similarity(e1, e2)
    print(f"Similarity('{w1}', '{w2}'): {sim:.4f}")

> **Question:** Look at the similarity scores. Are they in the order you expected? What pairs are most similar? Can you think of a word pair that would score even higher?

> **Note:** These are raw token embeddings before any Transformer layers. The scores reflect what the model learned during training, which can sometimes be surprising!


## 3. Attention Mechanism

**Self-attention** is what makes Transformers powerful. It allows each token to gather information from all other tokens in the sequence.

The formula for scaled dot-product attention is:

$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right) V$$

Where:
- **Q** (Query) — what this token is looking for
- **K** (Key)   — what each token offers / advertises
- **V** (Value) — the actual content to aggregate
- **d_k**       — dimension of the key vectors (used for scaling)

> **Discussion:** Think of Q/K/V like a search engine: Q is your search query, K is a list of document titles, and V is the document content. How does this analogy help explain what attention computes?


In [ ]:
def softmax(x):
    """Numerically stable softmax."""
    e = np.exp(x - np.max(x, axis=-1, keepdims=True))
    return e / e.sum(axis=-1, keepdims=True)

def scaled_dot_product_attention(Q, K, V):
    """
    Q: (seq_len, d_k)
    K: (seq_len, d_k)
    V: (seq_len, d_v)
    Returns: output (seq_len, d_v), attention_weights (seq_len, seq_len)
    """
    d_k = Q.shape[-1]
    # Step 1: compute raw attention scores
    scores = np.dot(Q, K.T) / np.sqrt(d_k)
    # Step 2: turn scores into probabilities with softmax
    weights = softmax(scores)
    # Step 3: weighted sum of value vectors
    output = np.dot(weights, V)
    return output, weights

In [ ]:
# Demo: 4-token sequence with d_k = d_v = 3
np.random.seed(42)
seq_len, d_k = 4, 3

Q = np.random.randn(seq_len, d_k)
K = np.random.randn(seq_len, d_k)
V = np.random.randn(seq_len, d_k)

output, weights = scaled_dot_product_attention(Q, K, V)

print("Attention weights (each row sums to 1):")
print(np.round(weights, 3))
print("\nOutput shape:", output.shape)

# Visualize attention weights as a heatmap
tokens_demo = ["The", "cat", "sat", "down"]
plt.figure(figsize=(5, 4))
plt.imshow(weights, cmap='Blues')
plt.colorbar()
plt.xticks(range(seq_len), tokens_demo)
plt.yticks(range(seq_len), tokens_demo)
plt.title("Attention Weight Heatmap")
plt.xlabel("Keys (attended to)")
plt.ylabel("Queries (attending)")
plt.tight_layout()
plt.show()

> **Question:** Look at the heatmap. Which token is "The" attending to most strongly? Does that make linguistic sense with random Q/K/V vectors?

> **Discussion:** In a real trained model, the Q, K, V matrices are learned. What kinds of linguistic patterns might a trained attention head discover? (e.g., subject-verb agreement, pronoun resolution)

> **Challenge:** What happens to the attention weights if you make all the Q and K vectors identical? Try it!


## 4. Text Generation with GPT-2

Now that we understand the building blocks, let's use a real pre-trained LLM — **GPT-2** — to generate text.

GPT-2 is an **autoregressive** model: it predicts the next token one at a time, appending each prediction to the input and repeating until the desired length is reached.

Key generation parameters:
- `max_new_tokens` — how many new tokens to generate
- `temperature` — controls randomness (lower = more deterministic, higher = more creative)
- `top_k` — only sample from the top-k most likely tokens
- `do_sample` — whether to sample or always pick the top token (greedy)

> **Discussion:** What do you think will happen if `temperature` is set very close to 0? Very high (e.g., 2.0)? What does "temperature" mean in the context of probability distributions?


In [ ]:
# Load a text-generation pipeline with GPT-2
generator = pipeline("text-generation", model="gpt2")

prompt = "Artificial intelligence is"
output = generator(
    prompt,
    max_new_tokens=60,
    do_sample=True,
    temperature=0.8,
    top_k=50,
    num_return_sequences=1
)

print(output[0]['generated_text'])

In [ ]:
# Try different temperatures to see the effect on creativity
prompt = "The future of machine learning"
temperatures = [0.2, 0.8, 1.5]

for temp in temperatures:
    result = generator(
        prompt,
        max_new_tokens=40,
        do_sample=True,
        temperature=temp,
        top_k=50,
        num_return_sequences=1
    )
    print(f"--- Temperature: {temp} ---")
    print(result[0]['generated_text'])
    print()

> **Question:** Compare the outputs at temperature 0.2 vs 1.5. Which one is more repetitive? Which one makes less sense? Is there a "sweet spot"?

> **Discussion:** GPT-2 was trained in 2019 and has only 117M parameters. How do you think modern LLMs (GPT-4, Claude, LLaMA 3) with billions of parameters differ in their outputs? What else beyond size might matter?


In [ ]:
# Your turn: try your own prompt!
my_prompt = "In a world where robots and humans"

my_output = generator(
    my_prompt,
    max_new_tokens=80,
    do_sample=True,
    temperature=1.0,
    top_k=40,
    num_return_sequences=2  # generate 2 different continuations
)

for i, out in enumerate(my_output):
    print(f"--- Continuation {i+1} ---")
    print(out['generated_text'])
    print()

> **Final Challenge:** Pick a prompt related to something you know well (your major, a hobby, a movie). Run the generator with two very different temperatures. Does GPT-2 "know" things about your topic, or does it hallucinate? What does this tell you about how LLMs store and retrieve knowledge?


## Summary

In this notebook we covered the four core concepts behind LLMs and Transformers:

1. **Tokenization** — text is split into subword tokens and mapped to integer IDs. GPT-2 has a vocabulary of 50,257 tokens.
2. **Embeddings** — each token ID maps to a 768-dimensional vector. Semantically similar words have similar vectors (high cosine similarity).
3. **Attention** — the key innovation of Transformers. Each token attends to all others via scaled dot-product attention: `softmax(QKᵀ / √d_k) · V`.
4. **Text Generation** — GPT-2 generates text autoregressively (one token at a time). Temperature controls creativity, top_k limits the sampling pool.

These building blocks form the foundation of every modern LLM — GPT-4, Claude, LLaMA, Gemini, and beyond.
